# NeuralEnsemble: independent fitted-model ensembles

`NeuralEnsemble` clones and independently fits any NAMpy neural regressor or
classifier, optionally on bootstrap samples. It is distinct from the jointly
trained `EnsembleTreeNAM` architecture.


## Ensemble in one view

$$
\widehat y(x)=\frac1M\sum_{m=1}^{M}\widehat y_m(x),
\qquad
s_t(x)=\operatorname{sd}_m\{f_{m,t}(x)\}.
$$

The first quantity is the ensemble prediction; $s_t$ measures between-member
variation of an additive term, not calibrated posterior uncertainty.

Independent refits can vary because of initialization, stochastic optimization,
data resampling, and early stopping. Averaging reduces variance when member
errors are not perfectly correlated. With `bootstrap=False`, diversity comes
from member seeds; with `bootstrap=True`, each member also receives a sampled
training set and aligned sample weights or offsets.

Regression predictions are averaged on the response scale. Additive terms,
links, and intercepts are averaged on the link scale so the mean decomposition
remains internally coherent. For classification, class probabilities are
averaged before selecting a label. The reported standard deviations summarize
between-member disagreement only: they are neither confidence intervals nor a
Bayesian posterior.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

plt.style.use("seaborn-v0_8-whitegrid")
COLORS = ["#2563EB", "#F97316", "#10B981", "#8B5CF6", "#EF4444"]

rng = np.random.default_rng(7)
n = 180
X = pd.DataFrame({
    "x1": rng.uniform(-1.0, 1.0, n),
    "x2": rng.normal(size=n),
    "group": rng.choice(["a", "b", "c"], size=n),
})
y = (
    np.sin(np.pi * X["x1"])
    + 0.35 * X["x2"] ** 2
    + 0.30 * (X["group"] == "b")
    + rng.normal(0.0, 0.12, n)
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7
)

# Set True to run the small fit and all fitted-model demonstrations.
RUN_TRAINING = bool(globals().get("RUN_TRAINING", False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), constrained_layout=True)
axes[0].scatter(X["x1"], y, s=22, alpha=0.65, color=COLORS[0], edgecolor="none")
axes[0].set(title="Response across x1", xlabel="x1", ylabel="y")

group_order = ["a", "b", "c"]
group_values = [y[X["group"].to_numpy() == level] for level in group_order]
boxes = axes[1].boxplot(group_values, tick_labels=group_order, patch_artist=True)
for patch, color in zip(boxes["boxes"], COLORS, strict=False):
    patch.set_facecolor(color)
axes[1].set(title="Response by group", xlabel="group", ylabel="y")
fig.suptitle("Synthetic mixed-feature example", fontweight="bold")
plt.show()
plt.close(fig)


In [ ]:
from nampy.models import NAMRegressor, NeuralEnsemble

base = NAMRegressor(layer_sizes=[24, 12], dropout=0.0)
model = NeuralEnsemble(
    base,
    n_estimators=3,
    random_state=7,
    n_jobs=1,
    bootstrap=True,
)
model.get_params(deep=False)

# The same wrapper accepts a neural classifier. LSS aggregation is rejected
# because distribution parameters require family-specific aggregation rules.
from nampy.models import NAMClassifier

classifier_ensemble = NeuralEnsemble(
    NAMClassifier(layer_sizes=[24, 12], dropout=0.0),
    n_estimators=3,
    random_state=7,
)


## Fit, predict, and inspect uncertainty

Fit parameters after `y` are forwarded to every cloned member. Each member owns
its preprocessing and fitted architecture.


In [ ]:
if RUN_TRAINING:
    model.fit(
        X_train, y_train,
        max_epochs=3,
        batch_size=64,
        logger=False,
        enable_progress_bar=False,
        enable_model_summary=False,
    )
    predictions = model.predict(X_test)
    r2 = model.score(X_test, y_test)
    components = model.predict_components(X_test)
    uncertainty = model.predict_component_uncertainty(X_test, center=True)
    components.validate_additive_reconstruction(rtol=1e-5, atol=1e-6)
    from nampy.explanations import explain_additive_prediction, term_importance_table

    explanation = explain_additive_prediction(X_test, components, max_bins=24)
    importance = term_importance_table(components)
    display({"R2": r2, "members": uncertainty.n_estimators})
    display(importance)
    display(explanation.head(12))
    display({name: values.mean() for name, values in uncertainty.term_std.items()})

    observed = np.asarray(y_test).reshape(-1)
    fitted = np.asarray(predictions).reshape(-1)
    member_predictions = np.stack(
        [member.predict(X_test) for member in model.estimators_], axis=0
    )
    term_disagreement = {
        name: float(np.mean(values)) for name, values in uncertainty.term_std.items()
    }

    fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), constrained_layout=True)
    axes[0].scatter(observed, fitted, s=26, alpha=0.7, color=COLORS[0])
    lo, hi = min(observed.min(), fitted.min()), max(observed.max(), fitted.max())
    axes[0].plot([lo, hi], [lo, hi], "--", color="#334155", linewidth=1)
    axes[0].set(title="Observed vs ensemble", xlabel="Observed", ylabel="Predicted")

    order = np.argsort(X_test["x1"].to_numpy())
    for member_index, member_values in enumerate(member_predictions):
        axes[1].plot(
            X_test["x1"].to_numpy()[order],
            np.asarray(member_values).reshape(-1)[order],
            color=COLORS[member_index],
            alpha=0.55,
            linewidth=1,
        )
    axes[1].plot(X_test["x1"].to_numpy()[order], fitted[order], color="#0F172A", linewidth=2.3, label="Mean")
    axes[1].set(title="Member variation", xlabel="x1", ylabel="Prediction")
    axes[1].legend(frameon=False)

    disagreement = pd.Series(term_disagreement).sort_values()
    axes[2].barh(disagreement.index, disagreement.values, color=COLORS[2])
    axes[2].set(title="Term disagreement", xlabel="Mean between-member SD")
    fig.suptitle("Ensemble fit and uncertainty", fontweight="bold")
    plt.show()
    plt.close(fig)

    model.estimators_[0].plot_terms(X_test, center=True, rug=True, pages=1)


## Interpretation

Use independent ensembles when prediction stability or between-fit variation
matters more than the cost of fitting several complete estimators. Bootstrap
members add diversity; their spread measures fit-to-fit disagreement rather
than calibrated uncertainty. The wrapper accepts regressors and classifiers,
with classification probabilities averaged before labels are selected.


## Reference

- Breiman (1996), *Bagging Predictors*, for the classical variance-reduction
  motivation. NAMpy's wrapper also supports independent non-bootstrap refits.
